### NOMEANDO VARIÁVEIS

In [0]:
# Guarda o nome do catalog na variável catalog, ta workspace pq a conta é free e já veio assim por padrão
catalog = "workspace"   

# Nomes que vamos dar aos três databases (schemas) 
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Guardando o caminho completo de cada schema (catalog + nome do schema).
# Criado para futuramente usar f"{bronze_schema}.tabela" ao invés de f"{catalog}.{bronze_schema_name}.tabela"
bronze_schema = f"{catalog}.{bronze_schema_name}"   # workspace.bronze
silver_schema = f"{catalog}.{silver_schema_name}"   # workspace.silver
gold_schema = f"{catalog}.{gold_schema_name}"       # workspace.gold

# Guarda o caminho do Volume onde estão os 5 CSVs importados
landing_path = "/Volumes/workspace/cinedata_landing/inputs"

# Print pra validar se deu certo :p
print(f"bronze_schema: {bronze_schema}")
print(f"landing_path: {landing_path}")

bronze_schema: workspace.bronze
landing_path: /Volumes/workspace/cinedata_landing/inputs


### CRIANDO DATABASE SE NAO EXISTIR

In [0]:
# Cria o database (schema) bronze se ele nao existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_schema}")

# Print pra confirmar
print(f"Database {bronze_schema} criado")

Database workspace.bronze criado


### CARREGANDO CSVs E CRIANDO TABELAS

In [0]:
# Criando variáveis com o caminho completo de cada CSV
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# Print pra confirmar
print(path_credits_and_tags)
print(path_movies_financials)
print(path_movies_info)
print(path_movies_metrics)
print(path_movies_reviews)

/Volumes/workspace/cinedata_landing/inputs/credits_and_tags_IMDB_TMDB.csv
/Volumes/workspace/cinedata_landing/inputs/movies_financials_IMDB_TMDB.csv
/Volumes/workspace/cinedata_landing/inputs/movies_info_TMDB_IMDB.csv
/Volumes/workspace/cinedata_landing/inputs/movies_metrics_IMDB_TMDB.csv
/Volumes/workspace/cinedata_landing/inputs/movies_reviews.csv


In [0]:
# df = DataFrame
# .read.csv() - Leitura do CSV
# header=True - Primeira linha do CSV é o cabeçalho
# inferSchema=True - Descobrir o tipo de dados das colunas (string, double...)

# Leitura pura dos CSVs (sem transformações de negócio)
df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True)

# Print pra confirmar a quatidade de linhas de cada dataframe
print(f"movies_info: {df_movies_info_raw.count()} linhas")
print(f"movies_financials: {df_movies_financials_raw.count()} linhas")
print(f"movies_metrics: {df_movies_metrics_raw.count()} linhas")
print(f"credits_and_tags: {df_credits_and_tags_raw.count()} linhas")
print(f"movies_reviews: {df_movies_reviews_raw.count()} linhas")

movies_info: 106930 linhas
movies_financials: 106165 linhas
movies_metrics: 107364 linhas
credits_and_tags: 106320 linhas
movies_reviews: 32412 linhas


In [0]:
# importando a função current_timestamp() (pega data e hora exata do momento que é executada)
from pyspark.sql.functions import current_timestamp

#   DataFrame \
#       Adicionando a nova coluna ingestion_datetime \
#       Diz que vai ser no formato Delta Lake e o modo Append (imposto no desafio) \
#       Salva o resultado como uma tabela no Databricks \

# .withColumn() - Adiciona uma coluna nova ao DataFrame
# .write.format() - Define o formato de gravação
# .mode() - Define o modo de gravação
# .saveAsTable() - Salva a tabela no Databricks

# ATENÇÃOOOO!!!!
#   - COMO É FEITO COM MODO APPEND:
#       SE RODAR VARIAS VEZES, TERÃO DADOS DUPLICADOS A CADA EXECUÇÃO
#   - COMO É FEITO COM MODO OVERWRITE:
#       SEMPRE APAGA E REESCREVE A TABELA, PERDENDO OS DADOS ANTERIORES
# LEMBRAR DE USAR "DROP TABLE" SE PRECISAR EXECUTAR DNV!
#        oq usar: "spark.sql(f"DROP TABLE IF EXISTS {bronze_schema}.tb_movies_info"

df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

# Print de confirmação e as 5 primeiras linhas
print("Tabelas Bronze gravadas com sucesso.")
display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))    # spark.table() - Lê a tabela normal

Tabelas Bronze gravadas com sucesso.


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-18T17:16:45.327Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-18T17:16:45.327Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-18T17:16:45.327Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-18T17:16:45.327Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-18T17:16:45.327Z


In [0]:
# Verificando a tabela movies_financials
display(spark.table(f"{bronze_schema}.tb_movies_financials").limit(5))

id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-09-20T02:31:00.214Z
299536,300000000,2052415039,2026-09-20T02:31:00.214Z
299534,356000000,2800000000,2026-09-20T02:31:00.214Z
475557,55000000,1074458282,2026-09-20T02:31:00.214Z
271110,250000000,Não Informado,2026-09-20T02:31:00.214Z


In [0]:
# Verificando a tabela credits_and_tags
display(spark.table(f"{bronze_schema}.tb_credits_and_tags").limit(5))

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-09-20T02:31:06.013Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-09-20T02:31:06.013Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-09-20T02:31:06.013Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-09-20T02:31:06.013Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-09-20T02:31:06.013Z


In [0]:
# Verificando a tabela movies_metrics
display(spark.table(f"{bronze_schema}.tb_movies_metrics").limit(5))

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-09-18T17:16:55.904Z
299536,"154,34",8.255,27713,8.4,1406782,2026-09-18T17:16:55.904Z
299534,91.756,8.263,23857,8.4,1484150,2026-09-18T17:16:55.904Z
475557,"54,522",8.168,23425,8.3,1723035,2026-09-18T17:16:55.904Z
271110,70.741,7.4,21541,7.8,947222,2026-09-18T17:16:55.904Z


In [0]:
# Verificando a tabela movies_reviews
display(spark.table(f"{bronze_schema}.tb_movies_reviews").limit(5))

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-09-18T17:17:03.359Z
637007,Lucas Reis 602,3.9,null,2026-09-18T17:17:03.359Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-09-18T17:17:03.359Z
413036,Gabriela Monteiro 401,7.5,null,2026-09-18T17:17:03.359Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-09-18T17:17:03.359Z


### DATA E API

In [0]:
# Importando datetime pra pegar data e delta para intervalo de tempo
from datetime import datetime, timedelta

# datetime.today() - Pega a data de hoje (hoje relativo ao dia da execução kkkkkk)
# timedelta(days=x) - Pega uma data e subtrai x dias (nesse caso, 6 dias)

#       ========== EXPLICAÇÃO DO 6 DIAS, TESTEI COM 7 DIAS E TAVA ME RETORNANDO 8 DIAS NA VERDADE ============
#   NO INTERVALO DE 11/09 ATE 18/09, EXISTEM OS DIAS 11, 12, 13, 14, 15, 16, 17, 18, TOTALIZANDO 8 DIAS
#   POR ISSO, TO USANDO "days=6", com days=6 consigo exatamente o intervalo de 7 dias

# Guarda as datas de início e fim
data_fim_default = datetime.today()
data_inicio_default = data_fim_default - timedelta(days=6)

# Cria os parâmetros (widgets) de data início e fim
# dbutils.widgets - Modulo pra criar parametros (widgets)
# .text() - DEfine o formato como texto
dbutils.widgets.text("data_inicio", data_inicio_default.strftime("%m-%d-%Y"), "Data Início (MM-DD-AAAA)")   
dbutils.widgets.text("data_fim", data_fim_default.strftime("%m-%d-%Y"), "Data Fim (MM-DD-AAAA)")

# Da get nos valores já formatados pelo strftime()
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# Print pra confirmar dnv
print(f"Data início: {data_inicio_formatada}")
print(f"Data fim: {data_fim_formatada}")

Data início: 09-13-2026
Data fim: 09-19-2026


In [0]:
# Importando a biblioteca requests pra fazer requests HTTP 
import requests

#           ===== DUVIDA TIRADA NA MONITORIA =====
# Aqui eu tive o cuidado de nao dar select somente nas duas datas e a cotação, mas pegar todas as chaves do JSON
#   Fiz isso pois na arquiterua medalhão a bronze é conhecida por tratar dados brutos, não processados
#   Assim eu faço esse papel de filtrar somente as colunas que me interessam na própria silver

# NA PROPRIA URL TEM A FUNÇÃO COTAÇÃO DOLAR PERDIDO
# CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)

# Alteração da url fornecida no pdf do desafio substituindo as datas pelas minhas datas widget
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo("
    f"dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$format=json"
)

response = requests.get(url)    # Faz chamada HTTP e guarda a resposta
dados_cotacao = response.json() # Guarda o JSON de resposta

# Print pra confirmar status code e conteúdo do JSON
print(f"Status da requisição: {response.status_code}\n")
print(dados_cotacao)

Status da requisição: 200

{'@odata.context': 'https://was-p.bcnet.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata$metadata#_CotacaoDolarPeriodo', 'value': [{'cotacaoCompra': 5.169, 'cotacaoVenda': 5.1696, 'dataHoraCotacao': '2026-09-14 13:10:08.144425'}, {'cotacaoCompra': 5.1484, 'cotacaoVenda': 5.149, 'dataHoraCotacao': '2026-09-15 13:09:19.199664'}, {'cotacaoCompra': 5.152, 'cotacaoVenda': 5.1527, 'dataHoraCotacao': '2026-09-16 13:05:30.35873'}, {'cotacaoCompra': 5.1515, 'cotacaoVenda': 5.1521, 'dataHoraCotacao': '2026-09-17 13:03:21.858212'}, {'cotacaoCompra': 5.1569, 'cotacaoVenda': 5.1575, 'dataHoraCotacao': '2026-09-18 13:03:34.742036'}]}


In [0]:
# Remove a tabela antiga (schema desatualizado) para recriar com todos os campos da API
#spark.sql(f"DROP TABLE IF EXISTS {bronze_schema}.tb_cotacao_dolar")

# Remove os widgets q guardam valores q nao autalizam com nova excecucao da cedula
#dbutils.widgets.removeAll()

In [0]:
# TABELA COTAÇÕES COM TODAS AS CHAVES DA API QUE SERÃO USADAS NA SILVER
# NESSE MOMENTO AINDA EXISTEM AS LACUNAS DOS DIAS DE FINAIS DE SEMANA E FERIADOS, ISSO SERÁ TRATADO NA SILVER
# COMO O BRONZE NO MEDALHÃO NAO DEVE TER DADOS TRATADOS, EU VEJO PREENCHER ESTAS LACUNAS COMO UM TIPO DE "TRATAMENTO", POIS DEIXAM DE ESTAR CRUS

# Cria uma lista q guarda o conteudo das chaves "value" (que são as cotações) do JSON
lista_cotacoes = dados_cotacao["value"]

# spark.createDataFrame() - Cria um dataframe se baseando na lista_cotacoes
df_cotacao_dolar_raw = spark.createDataFrame(lista_cotacoes)

# Cria uma coluna da data de hoje (do dia da execução) 
# Grava sem reescrever nada (append) e salva a tabela da API
df_cotacao_dolar_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

# Print de confirmação
print("Tabela bronze.tb_cotacao_dolar gravada com sucesso.")
display(spark.table(f"{bronze_schema}.tb_cotacao_dolar"))

Tabela bronze.tb_cotacao_dolar gravada com sucesso.


cotacaoCompra,cotacaoVenda,dataHoraCotacao,ingestion_datetime
5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-18T21:52:34.103Z
5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-18T21:52:34.103Z
5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-18T21:52:34.103Z
5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-18T21:52:34.103Z
5.1569,5.1575,2026-09-18 13:03:34.742036,2026-09-18T21:52:34.103Z
5.169,5.1696,2026-09-14 13:10:08.144425,2026-09-19T03:57:28.142Z
5.1484,5.149,2026-09-15 13:09:19.199664,2026-09-19T03:57:28.142Z
5.152,5.1527,2026-09-16 13:05:30.35873,2026-09-19T03:57:28.142Z
5.1515,5.1521,2026-09-17 13:03:21.858212,2026-09-19T03:57:28.142Z
5.1569,5.1575,2026-09-18 13:03:34.742036,2026-09-19T03:57:28.142Z
